In [ ]:
import os

# For MAC with M-Chip:
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

from robreg.transforms.matrices import get_affine
from robreg.image import map

import nibabel as nib
import torch

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'

print(f"Running on device: {device}")

In [ ]:
# Download a test image
SETUP_DIR='./'
imgfile = f"{SETUP_DIR}140_orig.mgz"
if not os.path.exists(imgfile):
    !curl -k https://surfer.nmr.mgh.harvard.edu/pub/data/tutorial_data/buckner_data/tutorial_subjs/140/mri/orig.mgz -o f"{SETUP_DIR}140_orig.mgz"
else:
    print(f"File {imgfile} already exists, skipping")

In [ ]:
# Load Image
img = nib.load(imgfile)
idata = torch.from_numpy(img.get_fdata()).float()

# Generate Matrix for Mapping
trans = torch.tensor([3.1,0.6,1.2])
rot   = torch.tensor([0.08,0.04,0.02])
A = get_affine(translation=trans, rotvec=rot)
print(f"Vox-to-Vox matrix for mapping:\n{A}")

# Map Image
mapped_data = map(idata, transform=A, is_torch_mat=False)

# Save Image
mapped = nib.MGHImage(mapped_data.squeeze().numpy(), img.affine, img.header)
mapped.to_filename(f"{SETUP_DIR}140_trg.mgz")


In [ ]:
# Register images
from robreg import register_pyramid

v2v = register_pyramid(f"{SETUP_DIR}140_orig.mgz", f"{SETUP_DIR}140_trg.mgz",
                       mapped_name=f"{SETUP_DIR}140_reg.mgz",
                       lta_name=f"{SETUP_DIR}140.lta",
                       return_v2v=True, device=device)

print(f"\nFinal Vox-to-Vox Matrix:\n{v2v}")

print(f"\nDifference to Applied Transform: {torch.linalg.norm(v2v-A)}")